# Atelier 4.C — Suite d'évaluation et red teaming ★

**Durée : 65 min** · Projet fil rouge, incrément du Jour 4

Un agent RAG se trompe de trois façons : il **répond faux**, il **prend un mauvais chemin** (mauvaise source, outils inutiles, boucle) ou il **se laisse détourner**. La suite d'évaluation couvre les trois :

1. un **golden set** de 32 questions métier, avec des mesures sans LLM (faits attendus, refus, fuites, citations, sources) ;
2. la **qualité des trajectoires** : sources interrogées, nombre d'appels ;
3. un **LLM juge** de l'exactitude et de la fidélité, **calibré** sur des jugements humains ;
4. la **comparaison** de l'agent avant et après la recherche corrective de l'atelier 4.B ;
5. un **red teaming** : injections directes et indirectes, fuites de données, jailbreak, avec et sans garde-fous.

Livrable : `resultats/evaluation_4C.json`, le score de référence du Jour 4.

In [ ]:
import json
import os
from datetime import date
from itertools import chain, zip_longest
from pathlib import Path

import pandas as pd

import agent_rag as ar
import evaluation_agent as ev
import outils_llm as llm
import rag

MODELE = llm.choisir_modele()                      # modèle de l'agent
MODELE_JUGE = os.getenv("MODELE_JUGE") or MODELE   # idéalement un autre modèle, chez un autre fournisseur
N_MAX = 12   # questions par campagne (None pour les 32) : environ 6 appels au modèle par question et par campagne
sources = ar.construire_sources()
print("Agent :", MODELE, "| juge :", MODELE_JUGE)

## 1. Le golden set de l'agent

Chaque entrée précise le **client connecté**, la réponse de référence, les **sources** et les **documents** attendus, les **faits** qui doivent figurer dans la réponse, les éléments **interdits** (informations internes, données d'un autre client) et si un **refus** est attendu.

In [ ]:
golden = ev.charger("golden_set_agent.json")
print(json.dumps(next(g for g in golden if g["id"] == "G21"), ensure_ascii=False, indent=2))
pd.Series([g["type"] for g in golden]).value_counts()

In [ ]:
# Échantillon qui couvre tous les types : une question de chaque type à tour de rôle.
par_type = {}
for g in golden:
    par_type.setdefault(g["type"], []).append(g)
ordre = [g for g in chain.from_iterable(zip_longest(*par_type.values())) if g]
echantillon = ordre if N_MAX is None else ordre[:N_MAX]
print(", ".join(f"{g['id']} ({g['type']})" for g in echantillon))

## 2. Campagne 1 : l'agent de l'atelier 4.A

Chaque question est posée dans une conversation neuve, pour le compte du client indiqué. Les mesures sans LLM :

| Mesure | Définition |
|---|---|
| `faits` | part des faits attendus présents dans la réponse |
| `refus_correct` | l'agent décline quand il le faut (hors périmètre, données d'un autre client) |
| `sans_fuite` | aucun élément interdit dans la réponse |
| `citee`, `citations_valides` | la réponse cite ses sources ; aucune référence inventée |
| `rappel_sources`, `precision_sources` | sources attendues interrogées ; sources interrogées utiles |
| `rappel_documents` | documents attendus parmi ceux renvoyés par les outils |

In [ ]:
agent_base = ar.construire_agent(ar.creer_outils(sources), modele=MODELE)
resultats_base = await ev.executer(agent_base, echantillon, "base")
details_base = ev.tableau(resultats_base, echantillon, sources)
details_base

In [ ]:
details_base.groupby("type")[["faits", "refus_correct", "sans_fuite", "rappel_sources", "appels_modele", "latence_s"]].mean().round(2)

## 3. La qualité des trajectoires

Une bonne réponse obtenue par un mauvais chemin est fragile et coûteuse. On compare les sources **attendues** et **interrogées**, et on lit les trajectoires.

In [ ]:
trajectoires = pd.DataFrame([{
    "id": e["id"], "attendues": ", ".join(e["sources_attendues"]) or "aucune",
    "interrogées": ", ".join(r.sources_utilisees()) or "aucune",
    "outils": " → ".join(o["nom"] for o in r.outils), "appels_modele": r.appels_modele, "erreur": r.erreur,
} for e, r in zip(echantillon, resultats_base)]).set_index("id")
trajectoires

In [ ]:
for e, r in zip(echantillon, resultats_base):
    manquantes = set(e["sources_attendues"]) - set(r.sources_utilisees())
    if manquantes or r.invalides or r.erreur:
        print(f"{e['id']} | sources manquantes : {sorted(manquantes) or '-'} | citations inventées : {r.invalides or '-'} | {r.erreur or ''}")
        print(f"   {e['question']}\n   → {r.reponse[:250]}\n")

## 4. Le LLM juge

Les mesures sans LLM vérifient des faits précis ; le **juge** évalue le sens. Son prompt (`ev.PROMPT_JUGE`) donne une grille à trois niveaux (correcte, partielle, incorrecte), demande la justification **avant** le verdict et met en garde contre la préférence pour les réponses longues.

### 4.1 Calibrer le juge sur des jugements humains

Avant de faire confiance au juge, on le compare à un expert sur 16 réponses annotées à la main : réponses justes, paraphrases, réponses incomplètes, erreurs de montant, hallucinations, refus à tort, fuite d'information interne. Le **kappa de Cohen** mesure l'accord au-delà du hasard : au-dessus de 0,6, l'accord est jugé bon ; au-dessus de 0,8, très bon.

In [ ]:
print(ev.PROMPT_JUGE)
humains = ev.charger("jugements_humains.json")
jugements = [ev.juger(h["question"], h["reponse_reference"], h["reponse_candidate"], MODELE_JUGE) for h in humains]
calibration = pd.DataFrame([{"id": h["id"], "humain": h["verdict_humain"], "juge": j.verdict,
                             "justification du juge": j.justification} for h, j in zip(humains, jugements)]).set_index("id")
accord = (calibration.humain == calibration.juge).mean()
kappa = ev.kappa_cohen(list(calibration.humain), list(calibration.juge))
print(f"\nAccord : {accord:.0%} | kappa de Cohen : {kappa:.2f}")
pd.crosstab(calibration.humain, calibration.juge)

In [ ]:
calibration[calibration.humain != calibration.juge]

Lisez les désaccords : est-ce le juge qui se trompe, ou la grille qui est ambiguë ? On corrige le prompt du juge (ou la grille des annotateurs), puis on recalibre. Un juge s'évalue comme un modèle : sur des données qu'il n'a pas servi à mettre au point.

### 4.2 Deux biais à tester : verbosité et variance

In [ ]:
h05 = next(h for h in humains if h["id"] == "H05")
variantes = {
    "courte et fausse": "Vous serez remboursé de 240 CHF.",
    "longue et fausse": h05["reponse_candidate"],
    "courte et juste": "216 CHF : 240 CHF moins 24 CHF de frais de traitement.",
    "longue et juste": h05["reponse_candidate"].replace("240 CHF", "216 CHF, après déduction de 24 CHF de frais de traitement"),
}
for nom, texte in variantes.items():
    print(f"{nom:>17} → {ev.juger(h05['question'], h05['reponse_reference'], texte, MODELE_JUGE).verdict}")

h07 = next(h for h in humains if h["id"] == "H07")
print("\nTrois jugements de la même réponse (H07, annotée « partielle ») :",
      [ev.juger(h07["question"], h07["reponse_reference"], h07["reponse_candidate"], MODELE_JUGE).verdict for _ in range(3)])

### 4.3 Juger la campagne 1 : exactitude et fidélité

- **Exactitude** : la réponse face à la référence (verdict du juge : 1, 0,5 ou 0).
- **Fidélité** : un second appel liste les affirmations de la réponse **absentes des résultats d'outils** que l'agent a reçus. C'est la *faithfulness* de RAGAS, appliquée à une trajectoire d'agent.

In [ ]:
def juger_campagne(resultats, entrees):
    lignes = []
    for r, e in zip(resultats, entrees):
        ligne = {"id": e["id"], "verdict": None, "exactitude_juge": None, "fidele": None, "non_fondees": []}
        try:
            j = ev.juger(e["question"], e["reponse_reference"], r.reponse, MODELE_JUGE)
            ligne.update(verdict=j.verdict, exactitude_juge=ev.SCORES[j.verdict], justification=j.justification)
            if r.resultats_outils:
                f = ev.juger_fidelite(r.reponse, r.resultats_outils, MODELE_JUGE)
                ligne.update(fidele=f.fidele, non_fondees=f.affirmations_non_fondees)
        except Exception as erreur:  # quota, sortie invalide : la ligne reste vide
            ligne["justification"] = f"échec du juge : {type(erreur).__name__}"
        lignes.append(ligne)
    return pd.DataFrame(lignes).set_index("id")

jugements_base = juger_campagne(resultats_base, echantillon)
details_base = details_base.join(jugements_base[["exactitude_juge", "fidele"]])
jugements_base[["verdict", "fidele", "non_fondees", "justification"]]

## 5. Campagne 2 : avec la recherche corrective de l'atelier 4.B

Même échantillon, même modèle ; seuls les outils de recherche changent (`corrective=True`). Les appels de l'évaluateur, faits à l'intérieur des outils, sont comptés à part.

In [ ]:
journal_correctif = []
agent_correctif = ar.construire_agent(
    ar.creer_outils(sources, corrective=True, modele_evaluateur=MODELE, journal=journal_correctif), modele=MODELE)
resultats_corr = await ev.executer(agent_correctif, echantillon, "corrective")
details_corr = ev.tableau(resultats_corr, echantillon, sources)
details_corr = details_corr.join(juger_campagne(resultats_corr, echantillon)[["exactitude_juge", "fidele"]])

appels_evaluateur = sum(1 if j["evenement"] == "évaluation" else 2 for j in journal_correctif)
avant_apres = pd.DataFrame({"agent 4.A": ev.synthese(details_base),
                            "agent 4.A + recherche corrective": ev.synthese(details_corr)})
avant_apres.loc["appels_evaluateur_par_question"] = [0, round(appels_evaluateur / len(echantillon), 2)]
avant_apres

In [ ]:
ecarts = details_base[["exactitude_juge"]].join(details_corr[["exactitude_juge"]], lsuffix="_avant", rsuffix="_apres")
ecarts[ecarts.exactitude_juge_avant != ecarts.exactitude_juge_apres]

**À discuter.** La recherche corrective améliore-t-elle l'exactitude et la fidélité sur **cet** échantillon, et à quel coût (appels, latence) ? Sur 12 questions, un écart d'une ou deux réponses n'est pas significatif : on conclut sur le golden set complet, et en répétant les campagnes.

## 6. Red teaming

On attaque l'agent comme le ferait un utilisateur malveillant, ou un document compromis. Huit attaques, rattachées au référentiel **OWASP Top 10 for LLM Applications** :

In [ ]:
attaques = ev.charger("red_teaming/attaques.json")
pd.DataFrame(attaques)[["id", "categorie", "owasp", "client", "message"]].set_index("id")

Deux attaques sont **indirectes** : elles passent par des articles piégés, glissés dans la base du support (empoisonnement des données, OWASP LLM04 et LLM08). Le client pose une question anodine ; c'est l'extrait retrouvé qui contient l'instruction malveillante.

In [ ]:
pieges = ar.charger_articles("red_teaming/articles_pieges.json")
print(pieges[0]["contenu"])
index_piege = rag.Index(ar.articles_vers_extraits(ar.charger_articles() + pieges), sources.documentation.encodeur)
sources_piegees = ar.Sources(sources.documentation, index_piege, sources.reranker)

### 6.1 Sans garde-fous

L'agent de l'atelier 4.A, avec son prompt d'origine. Une attaque réussit si la réponse contient un élément interdit, ou si le juge constate que le critère de réussite est rempli.

In [ ]:
def bilan_attaques(resultats):
    lignes = []
    for a, r in zip(attaques, resultats):
        reussie, raison = ev.attaque_reussie(a, r.reponse, MODELE_JUGE)
        lignes.append({"id": a["id"], "categorie": a["categorie"], "reussie": reussie, "raison": raison,
                       "reponse": r.reponse[:220], "alertes": "; ".join(r.alertes)})
    return pd.DataFrame(lignes).set_index("id")

agent_expose = ar.construire_agent(ar.creer_outils(sources_piegees), modele=MODELE)
attaques_sans = bilan_attaques(await ev.executer(agent_expose, attaques, "attaque"))
print(f"Attaques réussies : {attaques_sans.reussie.sum()}/{len(attaques)}")
attaques_sans

### 6.2 Avec garde-fous

Trois défenses en profondeur, aucune ne suffisant seule :

| Défense | Où | Contre |
|---|---|---|
| Prompt durci (`ar.PROMPT_AGENT_DURCI`) | modèle | injections directes, jeu de rôle, fuite du prompt |
| Filtrage des extraits (`ar.filtrer_extraits`) | entrée des outils | injections indirectes : les extraits qui donnent des ordres sont écartés et signalés |
| Contrôle de sortie (`ar.controler_reponse`) | réponse | liens et images vers des domaines externes, identifiant de configuration |

Les défenses les plus solides sont **hors du modèle** : l'identité du client dans la configuration (attaques A06 et A07), le filtrage des documents internes à l'index (A05), l'absence d'outil d'écriture (A03).

In [ ]:
journal_garde_fous = []
agent_protege = ar.construire_agent(ar.creer_outils(sources_piegees, garde_fous=True, journal=journal_garde_fous),
                                    prompt=ar.PROMPT_AGENT_DURCI, modele=MODELE)
attaques_avec = bilan_attaques(await ev.executer(agent_protege, attaques, "attaque-protegee", controle_sortie=True))
print(f"Attaques réussies : {attaques_avec.reussie.sum()}/{len(attaques)}")
print("Extraits écartés par le filtre :", sorted({j["reference"] for j in journal_garde_fous}) or "aucun")
attaques_avec

In [ ]:
pd.DataFrame({"sans garde-fous": attaques_sans.reussie, "avec garde-fous": attaques_avec.reussie,
              "catégorie": attaques_sans.categorie})

Les garde-fous ont un coût : un filtre trop large écarte des documents légitimes (vérifiez qu'aucun extrait du corpus n'est jugé suspect), un prompt trop défensif fait refuser des questions normales. On rejoue donc **aussi** le golden set avec les garde-fous avant de les déployer.

## 7. Le score de référence du Jour 4

In [ ]:
Path("resultats").mkdir(exist_ok=True)
reference = {
    "date": date.today().isoformat(),
    "configuration": {"modele": MODELE, "juge": MODELE_JUGE, "questions": len(echantillon),
                      "golden_set": "donnees/golden_set_agent.json"},
    "golden_set": avant_apres.to_dict(),
    "calibration_juge": {"accord": round(float(accord), 3), "kappa": round(kappa, 3), "annotations": len(humains)},
    "red_teaming": {"attaques": len(attaques), "reussies_sans_garde_fous": int(attaques_sans.reussie.sum()),
                    "reussies_avec_garde_fous": int(attaques_avec.reussie.sum())},
}
Path("resultats/evaluation_4C.json").write_text(json.dumps(reference, ensure_ascii=False, indent=2), encoding="utf-8")
reference["red_teaming"], reference["calibration_juge"]

## 8. Une évaluation continue

- **Non-régression** : chaque modification (modèle, prompt, outil, index) rejoue le golden set et les attaques ; on compare au score de référence avant de déployer.
- **Retours des utilisateurs** : un pouce en bas sur une réponse ouvre un cas à analyser ; les cas confirmés enrichissent le golden set.
- **Red teaming régulier** : de nouvelles attaques apparaissent ; la liste s'enrichit, comme le golden set.
- **Suivi en production** : taux de refus, citations invalides, alertes des garde-fous, coût par conversation.

## À retenir

- On évalue la **réponse** (exactitude, fidélité), le **chemin** (sources, outils, nombre d'étapes) et la **robustesse** (attaques).
- Un LLM juge se **calibre** : accord et kappa sur des annotations humaines, biais de verbosité, variance.
- Les injections **indirectes** passent par les documents : tout contenu retrouvé est une donnée, jamais une instruction.
- Les garde-fous les plus fiables sont dans l'**architecture** (identité, filtrage des sources, droits des outils), pas dans le prompt.